In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from google.colab import drive
from tqdm import tqdm
import dlib
#!pip install mediapipe
#import mediapipe as mp
import seaborn as sns
from scipy.fftpack import fft2, fftshift
from scipy import ndimage
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from PIL import Image
from time import time
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from skimage.feature import hog
from imutils import face_utils
import argparse
import imutils
import cv2
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 shape_predictor_68_face_landmarks.dat.bz2

!pip install insightface onnxruntime
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier


from insightface.app import FaceAnalysis
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

--2025-12-08 02:06:04--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2025-12-08 02:06:04--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  15.1MB/s    in 5.0s    

2025-12-08 02:06:10 (12.2 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing

In [ ]:
path = "drive/MyDrive/datasci281/"
train_dir = path + "train/"
test_dir = path + "test/"
emotion_names = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
df_train = pd.DataFrame(columns=['path', 'expression'])
df_test = pd.DataFrame(columns=['path', 'expression'])

for expression in os.listdir(train_dir):
    dir_ = train_dir + expression
    for img_name in os.listdir(dir_):
        img_path = dir_ + '/' + img_name
        df_train = pd.concat([df_train, pd.DataFrame({'path': [img_path], 'expression': [expression]})], ignore_index=True)
# df_train.head()
for expression in os.listdir(test_dir):
    dir_ = test_dir + expression
    for img_name in os.listdir(dir_):
        img_path = dir_ + '/' + img_name
        df_test = pd.concat([df_test, pd.DataFrame({'path': [img_path], 'expression': [expression]})], ignore_index=True)



In [ ]:
df_train.head()

,path,expression
0,drive/MyDrive/datasci281/train/angry/Training_...,angry
1,drive/MyDrive/datasci281/train/angry/Training_...,angry
2,drive/MyDrive/datasci281/train/angry/Training_...,angry
3,drive/MyDrive/datasci281/train/angry/Training_...,angry
4,drive/MyDrive/datasci281/train/angry/Training_...,angry


In [ ]:
print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (28709, 2)
Test shape: (7178, 2)


In [ ]:
df_train['expression'].value_counts()

,count
expression,
happy,7215
neutral,4965
sad,4830
fear,4097
angry,3995
surprise,3171
disgust,436


In [ ]:
df_test['expression'].value_counts()

,count
expression,
happy,1774
sad,1247
neutral,1233
fear,1024
angry,958
surprise,831
disgust,111


In [ ]:
#Ecode the labels
le = LabelEncoder()
df_train['label'] = le.fit_transform(df_train['expression'])
df_test['label']  = le.transform(df_test['expression'])

In [ ]:
df_train.head()

,path,expression,label
0,drive/MyDrive/datasci281/train/angry/Training_...,angry,0
1,drive/MyDrive/datasci281/train/angry/Training_...,angry,0
2,drive/MyDrive/datasci281/train/angry/Training_...,angry,0
3,drive/MyDrive/datasci281/train/angry/Training_...,angry,0
4,drive/MyDrive/datasci281/train/angry/Training_...,angry,0


In [ ]:

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(224, 224))


def normalize_landmarks(pts):
    pts_centered = pts - pts[30]  # nose tip
    eye_dist = np.linalg.norm(pts[36] - pts[45]) + 1e-6
    pts_norm = pts_centered / eye_dist
    return pts_norm.flatten()  # (136,)


def extract_landmarks_from_path(img_path, detector, predictor):
    img = cv2.imread(img_path)
    if img is None:
        raise RuntimeError(f"Could not load {img_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (96, 96))   # upscale from FER2013 (48×48)

    rects = detector(gray, 1)

    if len(rects) == 0:
        # fallback: whole image
        h, w = gray.shape
        rects = [dlib.rectangle(0, 0, w, h)]

    shape = predictor(gray, rects[0])

    pts = np.array(
        [[shape.part(i).x, shape.part(i).y] for i in range(68)],
        dtype=np.float32
    )

    return pts



def build_landmark_dataset(df, detector, predictor):
    X = []
    y = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Landmarks"):
        try:
            pts = extract_landmarks_from_path(row["path"], detector, predictor)
            feat = normalize_landmarks(pts)
        except:
            feat = np.zeros(136, dtype=np.float32)

        X.append(feat)
        y.append(row["expression"])

    return np.array(X, dtype=np.float32), np.array(y)


def load_fer2013_hog(data_dir, ppc=8, cpb=2, ori=9):
    X = []
    y = []
    image_paths = []

    for label, emotion in enumerate(emotion_names):
        emotion_path = os.path.join(data_dir, emotion)
        if not os.path.isdir(emotion_path):
            continue

        for f in tqdm(os.listdir(emotion_path), desc=emotion):
            path = os.path.join(emotion_path, f)
            img = Image.open(path).convert('L').resize((96, 96))
            arr = np.array(img)

            feat = hog(
                arr,
                orientations=ori,
                pixels_per_cell=(ppc, ppc),
                cells_per_block=(cpb, cpb),
                block_norm="L2-Hys",
                feature_vector=True
            )

            X.append(feat)
            y.append(label)
            image_paths.append(path)

    return np.array(X, dtype=np.float32), np.array(y), image_paths



def extract_insightface_embedding(path):
    img = cv2.imread(path)
    img = cv2.resize(gray, (96, 96))
    if img is None:
        return np.zeros(512, dtype=np.float32)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces = app.get(img)

    if len(faces) == 0:
        return np.zeros(512, dtype=np.float32)

    return np.array(faces[0].embedding, dtype=np.float32)


def subsample(X, y, n=5000):
    if len(X) <= n:
        return X, y
    idx = np.random.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 94384.08KB/s] 


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (224, 224)


# Training data

In [ ]:
X_hog, y, image_paths = load_fer2013_hog(train_dir) # add rrename vars

In [ ]:
# --- LANDMARKS ---
X_land, _ = build_landmark_dataset(df_test, detector, predictor)

In [ ]:
# --- INSIGHTFACE ---
X_emb = np.vstack([extract_insightface_embedding(p) for p in tqdm(image_paths)])

In [ ]:
# --- FINAL CONCATENATION ---
X_final = np.hstack([X_hog, X_land, X_emb])

print("Final Feature Shape:", X_final.shape)

In [ ]:
print("HOG", X_hog.shape)
print("Landmarks", X_land.shape)
print("Insightface", X_emb.shape )

In [ ]:
np.save("drive/MyDrive/datasci281/X_train_embeddings.npy", X_final)
np.save("drive/MyDrive/datasci281/y_train_embeddings.npy", y)
np.save("drive/MyDrive/datasci281/X_train_hog.npy", X_hog)
np.save("drive/MyDrive/datasci281/X_train_land.npy", X_land)
np.save("drive/MyDrive/datasci281/X_train_arcface.npy", X_emb)

# Test Data

In [ ]:
X_hog, y, image_paths = load_fer2013_hog(test_dir) # add rrename vars



neutral: 100%|██████████| 1233/1233 [00:23<00:00, 52.45it/s] 


In [ ]:
# --- LANDMARKS ---
X_land, _ = build_landmark_dataset(df_test, detector, predictor)



Landmarks: 100%|██████████| 7178/7178 [01:08<00:00, 104.06it/s]


In [ ]:
# --- INSIGHTFACE ---
X_emb = np.vstack([extract_insightface_embedding(p) for p in tqdm(image_paths)])



100%|██████████| 7178/7178 [17:13<00:00,  6.95it/s]


In [ ]:
# --- FINAL CONCATENATION ---
X_final = np.hstack([X_hog, X_land, X_emb])

print("Final Feature Shape:", X_final.shape)

Final Feature Shape: (7178, 1548)


In [ ]:
print("HOG", X_hog.shape)
print("Landmarks", X_land.shape)
print("Insightface", X_emb.shape )

HOG (7178, 900)
Landmarks (7178, 136)
Insightface (7178, 512)


In [ ]:
np.save("drive/MyDrive/datasci281/X_test_embeddings.npy", X_final)
np.save("drive/MyDrive/datasci281/y_test_embeddings.npy", y)
np.save("drive/MyDrive/datasci281/X_test_hog.npy", X_hog)
np.save("drive/MyDrive/datasci281/X_test_land.npy", X_land)
np.save("drive/MyDrive/datasci281/X_test_arcface.npy", X_emb)



## Summary

This was created as the getting of the embeddins for the landmarks and the arcface required a hugeamount of time and resources. Saving the embedings in a numpy allowed us to create hyperparamter tuning for various classifiers.